In [1]:
import shutil
import os

def copy_first_100_files(source_dir, destination_dir):
    # Get a list of all files in the source directory
    files = os.listdir(source_dir)
    
    # Sort files to ensure consistent order (optional)
    files.sort()
    
    # Copy only the first 100 files
    for i, file_name in enumerate(files):
        #if i >= 100:
        #    break
        full_file_name = os.path.join(source_dir, file_name)
        if os.path.isfile(full_file_name):
            #shutil.copy(full_file_name, destination_dir)
            print(f"Copied: {full_file_name}")

# Replace with your actual source and destination directories
source_directory = 'C:\\recent caltech\\journal 2 GA for segmentation\\images of dataset\\test masks\\test'
#destination_directory = 'C:\\recent caltech\\journal 2 GA for segmentation\\caltech full dataset  of all segmenters\\ground truth'
destination_directory = 'C:\\recent caltech\\journal 2 GA for segmentation\\caltech full dataset  of all segmenters\\pspnet'

# Call the function to copy the first 100 files
copy_first_100_files(source_directory, destination_directory)


Copied: C:\recent caltech\journal 2 GA for segmentation\images of dataset\test masks\test\Acadian_Flycatcher_0009_29155.png
Copied: C:\recent caltech\journal 2 GA for segmentation\images of dataset\test masks\test\Acadian_Flycatcher_0013_29232.png
Copied: C:\recent caltech\journal 2 GA for segmentation\images of dataset\test masks\test\Acadian_Flycatcher_0038_795616.png
Copied: C:\recent caltech\journal 2 GA for segmentation\images of dataset\test masks\test\Acadian_Flycatcher_0046_795596.png
Copied: C:\recent caltech\journal 2 GA for segmentation\images of dataset\test masks\test\Acadian_Flycatcher_0068_795590.png
Copied: C:\recent caltech\journal 2 GA for segmentation\images of dataset\test masks\test\Acadian_Flycatcher_0073_795628.png
Copied: C:\recent caltech\journal 2 GA for segmentation\images of dataset\test masks\test\American_Crow_0117_25090.png
Copied: C:\recent caltech\journal 2 GA for segmentation\images of dataset\test masks\test\American_Crow_0137_25221.png
Copied: C:\rec

In [2]:
#from unet_model_with_functions_of_blocks import build_unet
import os
import numpy as np
from matplotlib import pyplot as plt
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

#New generator with rotation and shear where interpolation that comes with rotation and shear are thresholded in masks. 
#This gives a binary mask rather than a mask with interpolated values. 
seed=24
batch_size= 32
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os
#os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

from tensorflow.keras.layers import Conv2D, BatchNormalization, Activation, MaxPool2D, Conv2DTranspose, Concatenate, Input

from tensorflow.keras.layers import AveragePooling2D, GlobalAveragePooling2D, UpSampling2D, Reshape, Dense

from tensorflow.keras.models import Model
from tensorflow.keras.applications import ResNet50
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.layers import UpSampling2D, Conv2D, Activation, BatchNormalization, Concatenate, AveragePooling2D
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.regularizers import l2
from tensorflow.keras.applications import ResNet50
import keras

In [21]:
import keras
def dice_metric(y_pred, y_true):
    intersection = K.sum(K.sum(K.abs(y_true * y_pred), axis=-1))
    union = K.sum(K.sum(K.abs(y_true) + K.abs(y_pred), axis=-1))
    # if y_pred.sum() == 0 and y_pred.sum() == 0:
    #     return 1.0

    return 2*intersection / union

def jaccard_distance_loss(y_true, y_pred, smooth=100):
    """
    Jaccard = (|X & Y|)/ (|X|+ |Y| - |X & Y|)
            = sum(|A*B|)/(sum(|A|)+sum(|B|)-sum(|A*B|))
    
    The jaccard distance loss is usefull for unbalanced datasets. This has been
    shifted so it converges on 0 and is smoothed to avoid exploding or disapearing
    gradient.
    
    Ref: https://en.wikipedia.org/wiki/Jaccard_index
    
    @url: https://gist.github.com/wassname/f1452b748efcbeb4cb9b1d059dce6f96
    @author: wassname
    """
    intersection = K.sum(K.sum(K.abs(y_true * y_pred), axis=-1))
    sum_ = K.sum(K.sum(K.abs(y_true) + K.abs(y_pred), axis=-1))
    jac = (intersection + smooth) / (sum_ - intersection + smooth)
    return (1 - jac) * smooth


model = keras.models.load_model("C:\\recent caltech\\journal 2 GA for segmentation\\all segmentation models\\Unet.h5", compile = False)
#model = keras.models.load_model('\\model.h5', compile = False)

METRICS =  ["accuracy",
           tf.keras.metrics.AUC(),
           tf.keras.metrics.SensitivityAtSpecificity(0.5),
           tf.keras.metrics.SpecificityAtSensitivity(0.5),
          dice_metric,
          jaccard_distance_loss]

model.compile(optimizer = keras.optimizers.Adam(learning_rate=1e-4),
              loss = 'binary_crossentropy',
              metrics = METRICS
             )

In [22]:
import os
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model
from PIL import Image
import cv2

def load_and_preprocess_image(img_path, target_size):
    # Load the image with PIL
    img = Image.open(img_path).convert("RGB")
    
    # Resize the image to match the model's expected input size
    img = img.resize(target_size)
    
    # Convert the image to a numpy array
    img_array = image.img_to_array(img)
    
    # Expand dimensions to match the expected input shape (batch_size, height, width, channels)
    img_array = np.expand_dims(img_array, axis=0)
    
    # Normalize the image (Optional based on your model's preprocessing needs)
    img_array /= 255.0
    
    return img_array


def convert_to_rgb(img_path, target_size):
    img = Image.open(img_path).convert("RGB")  # Convert grayscale to RGB
    img = img.resize(target_size)
    img_array = np.array(img)
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension
    return img_array


def predict_images_in_folder(model, folder_path, target_size):
    # List all files in the folder
    files = os.listdir(folder_path)
    print(len(files))
    for file_name in files:
        # Construct the full path to the image
        img_path = os.path.join(folder_path, file_name)
        
        if os.path.isfile(img_path):
            # Preprocess the image
            img_array = load_and_preprocess_image(img_path, target_size)
            
            # Use the model to make predictions
            predictions = model.predict(img_array)
            
            output_image = np.squeeze(predictions)  # Remove batch dimension if necessary

            # Ensure pixel values are in the correct range (0-255 for uint8)
            output_image = (output_image * 255).astype(np.uint8)

            # Save the image
            cv2.imwrite('C:\\recent caltech\\journal 2 GA for segmentation\\caltech full dataset  of all segmenters\\Unet\\'+str(file_name)+'.png', output_image)
            #print(f"Predictions for {file_name}: {predictions}")

# Load your trained model (update the path to your model)
#model = load_model('path/to/your/model.h5')

# Folder containing the images
folder_path = 'C:\\recent caltech\\journal 2 GA for segmentation\\caltech full dataset  of all segmenters\\original images\\'

# Set the target size that your model expects (e.g., (224, 224) for models like VGG16)
target_size = (224, 224)

# Run predictions on all images in the folder
predict_images_in_folder(model, folder_path, target_size)


590
1/1 [==============================] - 0s 47ms/step


1/1 [==============================] - 0s 53ms/step


1/1 [==============================] - 0s 47ms/step


1/1 [==============================] - 0s 63ms/step


In [16]:
#this is for PSPnet
import os
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model
from PIL import Image
import cv2

def load_and_preprocess_image(img_path, target_size):
    # Load the image with PIL
    img = Image.open(img_path).convert("RGB")
    
    # Resize the image to match the model's expected input size
    img = img.resize(target_size)
    
    # Convert the image to a numpy array
    img_array = image.img_to_array(img)
    
    # Expand dimensions to match the expected input shape (batch_size, height, width, channels)
    img_array = np.expand_dims(img_array, axis=0)
    
    # Normalize the image (Optional based on your model's preprocessing needs)
    img_array /= 255.0
    
    return img_array

def predict_images_in_folder(model, folder_path, target_size):
    # List all files in the folder
    files = os.listdir(folder_path)
    
    for file_name in files:
        # Construct the full path to the image
        img_path = os.path.join(folder_path, file_name)
        
        if os.path.isfile(img_path):
            # Preprocess the image
            img_array = load_and_preprocess_image(img_path, target_size)
            
            # Use the model to make predictions
            predictions = model.predict(img_array)
            
            output_image = np.squeeze(predictions)  # Remove batch dimension if necessary
            
            #output_image = tf.image.resize(output_image, (224,224,1))

            # Ensure pixel values are in the correct range (0-255 for uint8)
            output_image = (output_image * 255).astype(np.uint8)
            
            output_image = cv2.resize( output_image, (224,224))

            # Save the image
            cv2.imwrite('C:\\recent caltech\\journal 2 GA for segmentation\\caltech full dataset  of all segmenters\\PSPnet\\'+str(file_name)+'.png', output_image)
            #print(f"Predictions for {file_name}: {predictions}")

# Load your trained model (update the path to your model)
#model = load_model('path/to/your/model.h5')

# Folder containing the images
folder_path = 'C:\\recent caltech\\journal 2 GA for segmentation\\caltech full dataset  of all segmenters\\original images\\'

# Set the target size that your model expects (e.g., (224, 224) for models like VGG16)
target_size = (256, 256)

# Run predictions on all images in the folder
predict_images_in_folder(model, folder_path, target_size)


1/1 [==============================] - 0s 63ms/step


1/1 [==============================] - 0s 31ms/step


1/1 [==============================] - 0s 45ms/step


1/1 [==============================] - 0s 48ms/step
